In [13]:
import numpy as np
from aeon.classification import BaseClassifier
from aeon.transformations.collection.convolution_based import MultiRocket
from aeon.transformations.collection.convolution_based._hydra import HydraTransformer
from aeon.utils.validation import check_n_jobs
from aeon.transformations.collection.interval_based import QUANTTransformer
import numpy as np
import polars as pl
from aeon.classification.base import BaseClassifier
from aeon.classification.feature_based import (
    Catch22Classifier,
)
import os
from aeon.transformations.collection.convolution_based import Rocket
from aeon.datasets.tsc_datasets import univariate
from sklearn.base import clone
from aeon.classification.convolution_based import MultiRocketHydraClassifier
from aeon.classification.convolution_based import RocketClassifier
from sklearn.metrics import accuracy_score
from aeon.classification.interval_based import QUANTClassifier
from autotsc import utils, models
from tqdm import tqdm
from aeon.classification.feature_based import Catch22Classifier
from aeon.classification.interval_based import QUANTClassifier
from aeon.classification.shapelet_based import RDSTClassifier


In [14]:
write_dir = "experiments/automl_oof_predictions"
os.makedirs(write_dir, exist_ok=True)

In [15]:
class CrossValidationWrapper(BaseClassifier):
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.trained_models_ = []
        self.fit_time_ = []
        self.fit_time_mean_ = None
        self.predict_time_ = []
        self.predict_time_mean_ = None

    def _fit(self, X, y):
        raise NotImplementedError()

    def _predict_proba(self, X):
        predictions = []
        for model in self.trained_models_:
            proba = model.predict_proba(X)
            predictions.append(proba)
        avg_proba = np.mean(predictions, axis=0)
        return avg_proba

    def _predict(self, X):
        probas = self._predict_proba(X)
        predicted_indices = np.argmax(probas, axis=1)
        return self.classes_[predicted_indices]

    def get_all_oof_proba(self):
        return pl.DataFrame(self.oof_proba).sort('index', maintain_order=True)

    def _fit_predict_proba(self, X, y, cv_splits):
        self.oof_proba = []
        for train_idx, val_idx in tqdm(cv_splits):
            model_clone = clone(self.model)
            X_train, y_train = X[train_idx], y[train_idx]
            X_valid, _ = X[val_idx], y[val_idx]
            model_clone.fit(X_train, y_train)
            self.trained_models_.append(model_clone)
            proba = model_clone.predict_proba(X_valid)
            prob_columns = []
            for idx, p in zip(val_idx, proba):
                d = {
                    'index': idx,
                }
                for scls, prob in zip(model_clone.classes_, p):
                    k = f'proba_class_{scls}'
                    d[k] = prob.item()
                    if k not in prob_columns:
                        prob_columns.append(k)
                self.oof_proba.append(d)
        return pl.DataFrame(self.oof_proba).group_by('index').mean().sort('index').select(prob_columns).to_numpy()

    def fit_predict_proba(self, X, y, cv_splits):
        X, y, single_class = self._fit_setup(X, y)
        y_proba = self._fit_predict_proba(X, y, cv_splits)
        self.is_fitted = True
        return y_proba

In [19]:
def generate_folds(X, y, n_splits=5, n_repetitions=5, random_state=0):
    all_folds = []
    for i in range(n_repetitions):
        folds = utils.get_folds(X, y, n_splits=n_splits, random_state=random_state+i)
        all_folds.extend(folds)
    return all_folds

import numpy as np
from scipy.interpolate import interp1d
from aeon.transformations.collection.base import BaseCollectionTransformer


class DownsampleTransformer(BaseCollectionTransformer):
    _tags = {
        "X_inner_type": ["np-list", "numpy3D"],
        "capability:multivariate": True,
        "capability:unequal_length": True,
        "fit_is_empty": True,
    }

    def __init__(self, proportion):
        self.proportion = proportion
        super().__init__()

    def _transform(self, X, y=None):
        self._check_parameters()

        is_np = isinstance(X, np.ndarray)
        out = []

        for x in X:
            c, t = x.shape
            new_t = max(2, int(round(t * self.proportion)))

            old_grid = np.linspace(0, 1, t)
            new_grid = np.linspace(0, 1, new_t)

            xr = np.zeros((c, new_t))
            for i in range(c):
                f = interp1d(old_grid, x[i], kind="linear")
                xr[i] = f(new_grid)

            out.append(xr)

        return np.asarray(out) if is_np else out

    def _check_parameters(self):
        if not (0 < self.proportion < 1):
            raise ValueError("proportion must be between 0 and 1")


In [20]:
dataset = 'Lightning7'
X_train, y_train, X_test, y_test = utils.load_dataset(dataset)

In [21]:
all_folds = generate_folds(X_train, y_train, n_splits=10, n_repetitions=10, random_state=0)

/home/gasper_p/workspace/repos/AutoTSC/.venv/lib/python3.12/site-packages/sklearn/model_selection/_split.py:811: UserWarning: The least populated class in y has only 5 members, which is less than n_splits=10.
  warnings.warn(
/home/gasper_p/workspace/repos/AutoTSC/.venv/lib/python3.12/site-packages/sklearn/model_selection/_split.py:811: UserWarning: The least populated class in y has only 5 members, which is less than n_splits=10.
  warnings.warn(
/home/gasper_p/workspace/repos/AutoTSC/.venv/lib/python3.12/site-packages/sklearn/model_selection/_split.py:811: UserWarning: The least populated class in y has only 5 members, which is less than n_splits=10.
  warnings.warn(
/home/gasper_p/workspace/repos/AutoTSC/.venv/lib/python3.12/site-packages/sklearn/model_selection/_split.py:811: UserWarning: The least populated class in y has only 5 members, which is less than n_splits=10.
  warnings.warn(
/home/gasper_p/workspace/repos/AutoTSC/.venv/lib/python3.12/site-packages/sklearn/model_selectio

In [22]:
m1 = CrossValidationWrapper(MultiRocketHydraClassifier(n_jobs=-1, random_state=0))
prob_hr = m1.fit_predict_proba(X_train, y_train, all_folds)

100%|██████████| 100/100 [02:20<00:00,  1.41s/it]


In [23]:
m2 = MultiRocketHydraClassifier(n_jobs=-1, random_state=0)
m2.fit(X_train, y_train)

,n_kernels,8
,n_groups,64
,class_weight,None
,n_jobs,-1
,random_state,0


In [24]:
pred_m1 = m1._predict(X_train)
pred_m2 = m2.predict(X_train)

acc_m1 = accuracy_score(y_train, pred_m1)
acc_m2 = accuracy_score(y_train, pred_m2)
print(acc_m1, acc_m2)

1.0 1.0


In [3]:
def fold_generator(X, y, n_fold=10, random_state=None):
    from sklearn.model_selection import StratifiedKFold, KFold
    rng = np.random.default_rng(random_state)
    while True:
        seed = int(rng.integers(0, 2**32 - 1))

        try:
            skf = StratifiedKFold(
                n_splits=n_fold,
                shuffle=True,
                random_state=seed
            )
            for i, (train_idx, val_idx) in enumerate(skf.split(X, y)):
                split_id = f'SKF{skf.random_state}.{i}'
                yield split_id, train_idx, val_idx

        except ValueError:
            kf = KFold(
                n_splits=n_fold,
                shuffle=True,
                random_state=seed
            )
            for i, (train_idx, val_idx) in enumerate(kf.split(X)):
                split_id = f'KF{skf.random_state}.{i}'
                yield split_id, train_idx, val_idx

dataset = 'Lightning7'
X_train, y_train, X_test, y_test = utils.load_dataset(dataset)

from itertools import islice

run = 0


for split_id, train_idx, val_idx in islice(fold_generator(X_train, y_train, random_state=run), 100):
    model_name = 'rocket'

    stats = {
        'dataset': dataset,
        'model': model_name,
        'split_id': split_id,
        'run': run,
    }

    hash_val = pl.DataFrame([stats]).hash_rows(seed=42, seed_1=1, seed_2=2, seed_3=3).item()
    file = f"{write_dir}/{hash_val}.parquet"

    if os.path.exists(file):
        continue

    model = RocketClassifier(random_state=run, n_jobs=-1)
    model.fit(X_train[train_idx], y_train[train_idx])
    y_pred = model.predict(X_train[val_idx])
    y_pred_proba = model.predict_proba(X_train[val_idx])

    stats['accuracy'] = accuracy_score(y_train[val_idx], y_pred)
    stats['predictions'] = y_pred.tolist()
    stats['true'] = y_train[val_idx].tolist()
    stats['classes'] = model.classes_.tolist()
    stats['predictions_proba'] = y_pred_proba.tolist()
    stats['train_idx'] = train_idx.tolist()
    stats['val_idx'] = val_idx.tolist()



    df_stat = pl.DataFrame([stats])
    df_stat.write_parquet(file, mkdir=True)

/home/gasper_p/workspace/repos/AutoTSC/.venv/lib/python3.12/site-packages/sklearn/model_selection/_split.py:811: UserWarning: The least populated class in y has only 5 members, which is less than n_splits=10.
  warnings.warn(
/home/gasper_p/workspace/repos/AutoTSC/.venv/lib/python3.12/site-packages/sklearn/model_selection/_split.py:811: UserWarning: The least populated class in y has only 5 members, which is less than n_splits=10.
  warnings.warn(
/home/gasper_p/workspace/repos/AutoTSC/.venv/lib/python3.12/site-packages/sklearn/model_selection/_split.py:811: UserWarning: The least populated class in y has only 5 members, which is less than n_splits=10.
  warnings.warn(
/home/gasper_p/workspace/repos/AutoTSC/.venv/lib/python3.12/site-packages/sklearn/model_selection/_split.py:811: UserWarning: The least populated class in y has only 5 members, which is less than n_splits=10.
  warnings.warn(
/home/gasper_p/workspace/repos/AutoTSC/.venv/lib/python3.12/site-packages/sklearn/model_selectio

In [4]:
df = pl.read_parquet(f'{write_dir}/*.parquet')
df

dataset,model,split_id,run,accuracy,predictions,true,classes,predictions_proba,train_idx,val_idx
str,str,str,i64,f64,list[str],list[str],list[str],list[list[f64]],list[i64],list[i64]
"""Lightning7""","""rocket""","""SKF70985653.5""",0,0.857143,"[""2"", ""3"", … ""5""]","[""2"", ""3"", … ""5""]","[""0"", ""1"", … ""6""]","[[0.0, 0.0, … 0.0], [0.0, 0.0, … 0.0], … [0.0, 0.0, … 0.0]]","[0, 1, … 69]","[20, 23, … 67]"
"""Lightning7""","""rocket""","""SKF3653403230.6""",0,0.857143,"[""6"", ""5"", … ""5""]","[""6"", ""1"", … ""5""]","[""0"", ""1"", … ""6""]","[[0.0, 0.0, … 1.0], [0.0, 0.0, … 0.0], … [0.0, 0.0, … 0.0]]","[0, 1, … 69]","[5, 15, … 66]"
"""Lightning7""","""rocket""","""SKF3492969079.5""",0,0.857143,"[""2"", ""3"", … ""5""]","[""2"", ""3"", … ""0""]","[""0"", ""1"", … ""6""]","[[0.0, 0.0, … 0.0], [0.0, 0.0, … 0.0], … [0.0, 0.0, … 0.0]]","[0, 1, … 69]","[8, 10, … 61]"
"""Lightning7""","""rocket""","""SKF1158725111.6""",0,1.0,"[""2"", ""5"", … ""6""]","[""2"", ""5"", … ""6""]","[""0"", ""1"", … ""6""]","[[0.0, 0.0, … 0.0], [0.0, 0.0, … 0.0], … [0.0, 0.0, … 1.0]]","[0, 1, … 69]","[21, 28, … 64]"
"""Lightning7""","""rocket""","""SKF1158725111.5""",0,0.857143,"[""5"", ""3"", … ""5""]","[""5"", ""3"", … ""6""]","[""0"", ""1"", … ""6""]","[[0.0, 0.0, … 0.0], [0.0, 0.0, … 0.0], … [0.0, 0.0, … 0.0]]","[0, 2, … 69]","[1, 10, … 44]"
…,…,…,…,…,…,…,…,…,…,…
"""Lightning7""","""rocket""","""SKF3653403230.4""",0,0.857143,"[""0"", ""3"", … ""1""]","[""0"", ""3"", … ""1""]","[""0"", ""1"", … ""6""]","[[1.0, 0.0, … 0.0], [0.0, 0.0, … 0.0], … [0.0, 1.0, … 0.0]]","[0, 1, … 69]","[2, 3, … 46]"
"""Lightning7""","""rocket""","""SKF175979944.3""",0,0.714286,"[""0"", ""6"", … ""4""]","[""0"", ""6"", … ""1""]","[""0"", ""1"", … ""6""]","[[1.0, 0.0, … 0.0], [0.0, 0.0, … 1.0], … [0.0, 0.0, … 0.0]]","[0, 1, … 69]","[2, 5, … 57]"
"""Lightning7""","""rocket""","""SKF2735729614.4""",0,1.0,"[""2"", ""6"", … ""5""]","[""2"", ""6"", … ""5""]","[""0"", ""1"", … ""6""]","[[0.0, 0.0, … 0.0], [0.0, 0.0, … 1.0], … [0.0, 0.0, … 0.0]]","[0, 1, … 69]","[21, 22, … 66]"


In [5]:
v = {}

for row in df.iter_rows(named=True):
    a = row['predictions_proba']
    b = row['val_idx']
    for indx, pv in zip(b, a):
        #print(indx, pv)
        if indx not in v:
            v[indx] = []
        v[indx].append(pv)

In [6]:
np.unique(y_train)

array(['0', '1', '2', '3', '4', '5', '6'], dtype='<U1')

In [7]:
for k in sorted(list(v.keys())):
    print(k, np.array(v[k]).mean(axis=0), y_train[k])

0 [0. 0. 0. 0. 0. 1. 0.] 5
1 [0. 0. 0. 0. 0. 1. 0.] 5
2 [1. 0. 0. 0. 0. 0. 0.] 0
3 [0. 0. 0. 1. 0. 0. 0.] 3
4 [0. 0. 0. 0. 1. 0. 0.] 4
5 [0. 0. 0. 0. 0. 0. 1.] 6
6 [0. 0. 1. 0. 0. 0. 0.] 2
7 [1. 0. 0. 0. 0. 0. 0.] 0
8 [0. 0. 1. 0. 0. 0. 0.] 2
9 [0. 0. 0. 1. 0. 0. 0.] 3
10 [0. 0. 0. 1. 0. 0. 0.] 3
11 [0. 0. 0. 0. 0. 1. 0.] 5
12 [0. 0. 1. 0. 0. 0. 0.] 2
13 [0. 0. 1. 0. 0. 0. 0.] 5
14 [0. 0. 0. 0. 0. 1. 0.] 5
15 [0.  0.5 0.  0.  0.  0.5 0. ] 1
16 [0. 0. 0. 0. 0. 0. 1.] 6
17 [0. 0. 1. 0. 0. 0. 0.] 3
18 [0. 0. 0. 1. 0. 0. 0.] 3
19 [0. 0. 0. 0. 0. 1. 0.] 5
20 [0. 0. 1. 0. 0. 0. 0.] 2
21 [0. 0. 1. 0. 0. 0. 0.] 2
22 [0. 0. 0. 0. 0. 0. 1.] 6
23 [0. 0. 0. 1. 0. 0. 0.] 3
24 [0.  0.6 0.3 0.  0.  0.  0.1] 1
25 [0. 0. 0. 0. 0. 0. 1.] 6
26 [0. 0. 0. 0. 0. 1. 0.] 5
27 [1. 0. 0. 0. 0. 0. 0.] 0
28 [0. 0. 0. 0. 0. 1. 0.] 5
29 [0. 0. 0. 0. 0. 1. 0.] 5
30 [0. 0. 1. 0. 0. 0. 0.] 2
31 [0. 1. 0. 0. 0. 0. 0.] 1
32 [0. 0. 0. 1. 0. 0. 0.] 3
33 [0. 0. 0. 0. 0. 0. 1.] 6
34 [0.  0.  0.  0.4 0.6 0.  0. ] 4
35 [1. 0.